# 03 — Fitur, Perbandingan Algoritma, dan Model RUL (Orang 2)

Notebook ini punya dua bagian.

- **Bagian A — Perbandingan algoritma.** Lima kandidat dilatih di kelompok A, dipilih di B, dan dilaporkan di test resmi. Hasilnya diekspor ke `data/results/rul_evaluasi.json` untuk Orang 3 (bagian `rul` di `evaluasi.json`).
- **Bagian B — Model final.** Artefak `models/v1/` (Kontrak 5) dan `data/results/predictions.csv` untuk kelompok B dan C (bahan RL Orang 3).

Semua logika ada di `engine/` (`features.py`, `rul.py`, `predict.py`); notebook hanya memanggilnya, sehingga angka di sini sama dengan angka di server.

In [ ]:
# Colab: jalankan sel ini dulu. Sesuaikan PATH_DRIVE dan BRANCH.
import os, shutil, sys
from google.colab import drive
drive.mount('/content/drive')
PATH_DRIVE = '/content/drive/MyDrive/Praktikum MLN/Dataset ML'
BRANCH = 'model'
if not os.path.exists('/content/prognosense'):
    !git clone https://github.com/Arsha8Anuga/PrognoSense /content/prognosense
%cd /content/prognosense
!git checkout {BRANCH} 2>/dev/null || git checkout -b {BRANCH}
sys.path.insert(0, '/content/prognosense')
os.makedirs('data/raw', exist_ok=True)
for f in ['train_FD002.txt', 'train_FD004.txt', 'test_FD002.txt', 'test_FD004.txt', 'RUL_FD002.txt', 'RUL_FD004.txt']:
    shutil.copy(f'{PATH_DRIVE}/{f}', f'data/raw/{f}')
!pip -q install lightgbm

## 1. Memuat data dan pembagian mesin

Loader dan daftar kolom memakai `engine.data` dan `engine.schema` milik Orang 1. Mesin test diberi ID `FD002_T1`, `FD004_T1`, dst. supaya tidak bentrok dengan ID train (Kontrak 3).

In [ ]:
import json, os, sys, time, platform
from datetime import datetime, timezone, timedelta
from importlib.metadata import version
import numpy as np, pandas as pd, joblib
from sklearn.cluster import KMeans

from engine.schema import SENSOR_AKTIF, SETTING
from engine.data import load_and_format, load_splits
from engine.conditions import assign_kondisi, apply_normalization
from engine.features import buat_fitur, label_rul, nama_fitur, RUL_CAP
from engine.health import NAMA_TAHAP
from engine import rul as R

RAW, HASIL, MODEL = 'data/raw', 'data/results', 'models/v1'
os.makedirs(MODEL, exist_ok=True)

train = pd.concat([load_and_format(f'{RAW}/train_FD002.txt', 'FD002'),
                   load_and_format(f'{RAW}/train_FD004.txt', 'FD004')], ignore_index=True)
test = pd.concat([load_and_format(f'{RAW}/test_FD002.txt', 'FD002_T'),
                  load_and_format(f'{RAW}/test_FD004.txt', 'FD004_T')], ignore_index=True)
test['unit'] = test['unit'].str.replace('_T_', '_T')          # FD002_T_1 -> FD002_T1 (Kontrak 3)
splits = load_splits('data/splits.json')
kelompok = {u: k for k in 'ABC' for u in splits[k]}
train['kelompok'] = train['unit'].map(kelompok)
assert train['kelompok'].notna().all(), 'ada mesin train yang tidak masuk splits.json'
print(train.shape, test.shape, train.groupby('kelompok')['unit'].nunique().to_dict())

## 2. Kondisi operasi dan normalisasi

Kalau `models/v1/kmeans_cond.joblib` dan `norm_stats.json` dari Orang 1 sudah ada, keduanya dipakai. Kalau belum, keduanya direproduksi dengan parameter Kontrak 3 (K = 6, `n_init = 10`, `random_state = 42`, statistik **hanya dari kelompok A**).

Dua `assert` memastikan hasilnya sama dengan `kondisi.json` milik Orang 1: centroid beserta urutan labelnya, dan nilai `s11` ternormalisasi untuk mesin contoh. Kalau salah satu gagal, hampir pasti versi scikit-learn berbeda, dan itu harus diketahui sekarang, bukan di server.

In [ ]:
A = train[train['kelompok'] == 'A']
if os.path.exists(f'{MODEL}/kmeans_cond.joblib'):
    km_cond = joblib.load(f'{MODEL}/kmeans_cond.joblib')
    norm_stats = json.load(open(f'{MODEL}/norm_stats.json'))
    print('memakai artefak Orang 1 yang sudah ada')
else:
    km_cond = KMeans(n_clusters=6, n_init=10, random_state=42).fit(A[SETTING])
    kond_A = km_cond.predict(A[SETTING])
    norm_stats = {str(c): {s: {'mean': float(A.loc[kond_A == c, s].mean()), 'std': float(A.loc[kond_A == c, s].std())}
                           for s in SENSOR_AKTIF} for c in range(6)}
    joblib.dump(km_cond, f'{MODEL}/kmeans_cond.joblib')
    json.dump(norm_stats, open(f'{MODEL}/norm_stats.json', 'w'), indent=2)
    print('direproduksi dari kelompok A dan disimpan')

ref = json.load(open(f'{HASIL}/kondisi.json'))
selisih = np.abs(km_cond.cluster_centers_ - np.array(ref['centroid'])).max()
assert selisih < 1e-3, f'centroid atau urutan label berbeda dari kondisi.json ({selisih}); cek versi scikit-learn'
train = apply_normalization(assign_kondisi(train, km_cond), norm_stats, SENSOR_AKTIF)
test = apply_normalization(assign_kondisi(test, km_cond), norm_stats, SENSOR_AKTIF)
ss = ref['sebelum_sesudah']
cek = train.loc[train['unit'] == ss['unit'], f"{ss['sensor']}_norm"].values[:5]
assert np.allclose(cek, ss['normal'][:5], atol=1e-4), 'normalisasi berbeda dari kondisi.json'
print('centroid dan normalisasi cocok dengan kondisi.json | std terkecil:',
      round(min(v['std'] for c in norm_stats.values() for v in c.values()), 3))

## 3. Fitur jendela 10 siklus dan label RUL

`buat_fitur` membuat 43 fitur: `_last`, `_mean10`, `_slope10` untuk 14 sensor, ditambah `cycle`. Alasannya dari data: noise per siklus kira-kira 27 kali lebih besar dari kenaikan degradasi per siklus, sehingga rata-rata dan kemiringan jendela membawa sinyal yang tidak terlihat dari satu titik.

Label `RUL = siklus_terakhir − cycle`, dipotong di 125, karena di atas itu sensor hanya naik landai dan tenggelam dalam noise.

In [ ]:
F = buat_fitur(train)
F['rul'] = label_rul(F)
F['kelompok'] = F['unit'].map(kelompok)
FT = buat_fitur(test)
X = nama_fitur()
assert len(X) == 43 and not F[X].isna().any().any() and not FT[X].isna().any().any()
fA, fB = F[F['kelompok'] == 'A'], F[F['kelompok'] == 'B']
print('baris latih A:', len(fA), '| validasi B:', len(fB))

## 4. Test resmi: hanya siklus terakhir

`RUL_FD00x.txt` hanya memberi jawaban untuk siklus terakhir setiap mesin test, jadi penilaian test memakai 507 titik, satu per mesin, dengan RUL benar ikut dipotong di 125.

In [ ]:
y_test = pd.concat([R.rul_test_resmi(f'{RAW}/RUL_FD002.txt', 'FD002_T'),
                    R.rul_test_resmi(f'{RAW}/RUL_FD004.txt', 'FD004_T')])
ft_akhir = FT.groupby('unit').tail(1).set_index('unit').loc[y_test.index]
print('mesin test:', len(ft_akhir))

---
# Bagian A — Perbandingan algoritma

Aturan main: **latih di A, pilih di B, laporkan di test.** Test tidak pernah dipakai untuk memilih apa pun.

## 5. Lima kandidat

| Kandidat | Kenapa diikutkan |
|---|---|
| Baseline rata-rata | Batas bawah: model yang tidak mengalahkannya tidak belajar apa pun. |
| Regresi linear (Ridge) | Model paling sederhana; menguji apakah hubungan sensor–RUL cukup lurus. |
| kNN (k = 15) | "Cari mesin yang paling mirip"; tanpa asumsi bentuk, tetapi berat saat menebak. |
| Random Forest | Banyak pohon independen dirata-rata; stabil. |
| LightGBM | Pohon berurutan yang memperbaiki sisa error; cepat. |

Waktu latih dan waktu menebak dicatat terpisah. kNN hampir tidak butuh waktu latih karena ia hanya menyimpan data; biayanya muncul saat menebak.

In [ ]:
hasil, model = [], {}
for nama, m in R.kandidat().items():
    t0 = time.time(); m.fit(fA[X], fA['rul']); t1 = time.time(); pB = R.tebak(m, fB[X]); t2 = time.time()
    model[nama] = m
    b, t = R.skor(fB['rul'], pB), R.skor(y_test, R.tebak(m, ft_akhir[X]))
    hasil.append({'nama': nama, 'rmse_b': b['rmse'], 'mae_b': b['mae'], 'rmse': t['rmse'], 'mae': t['mae'],
                  'detik_latih': round(t1 - t0, 1), 'detik_tebak_b': round(t2 - t1, 1)})
    print(f"{nama:24s} B {b['rmse']:6.2f}  test {t['rmse']:6.2f}  latih {t1 - t0:5.1f} s  tebak B {t2 - t1:4.1f} s")
tabel = pd.DataFrame(hasil).set_index('nama')
tabel

## 6. Memilih model

Model dipilih dengan RMSE terkecil di **B**. Pada jalan acuan, LightGBM (15,72) dan Random Forest (15,81) praktis seri, tetapi LightGBM dilatih sekitar 28 kali lebih cepat. Regresi linear tertinggal (19,97) karena tidak bisa membentuk dataran RUL = 125; prediksinya keluar ke −36 dan 203.

Kalau Random Forest ternyata lebih baik di test, itu **ditulis di laporan, bukan dipakai untuk mengganti model.**

In [ ]:
terpilih = tabel['rmse_b'].idxmin()          # dipilih di B, bukan di test
m = model[terpilih]
print('model terpilih:', terpilih)

## 7. Ekspor untuk Orang 3

`data/results/rul_evaluasi.json` berisi tabel perbandingan, 507 titik prediksi vs aktual, dan 10 fitur terpenting. Bentuknya mengikuti bagian `rul` di `evaluasi.json` (Kontrak 6), ditambah kolom `rmse_b`, `mae_b`, dan waktu.

In [ ]:
imp = pd.Series(m.booster_.feature_importance('gain') if hasattr(m, 'booster_') else m.feature_importances_, index=X)
imp = (imp / imp.sum()).sort_values(ascending=False).head(10)
sebar = pd.DataFrame({'rul_aktual': y_test.values, 'rul_pred': R.tebak(m, ft_akhir[X]).round(1)})
rul_eval = {'terpilih': terpilih, 'dipilih_dengan': 'RMSE terkecil di kelompok B',
            'dinilai_pada': 'siklus terakhir tiap mesin test resmi FD002+FD004, RUL dipotong di 125',
            'model': tabel.reset_index().to_dict(orient='records'),
            'sebar': sebar.to_dict(orient='records'),
            'importance': [{'fitur': k, 'nilai': round(float(v), 4)} for k, v in imp.items()]}
json.dump(rul_eval, open(f'{HASIL}/rul_evaluasi.json', 'w'), indent=1)
print(imp.round(3))

---
# Bagian B — Model final

## 8. Tahap kesehatan

`health_kmeans.joblib` dipakai bila sudah ada dari Orang 1; kalau belum, direproduksi dengan prosedur yang sama. `engine.health` menganggap cluster 0/1/2 = sehat/menurun/kritis, jadi `assert` di sini memastikan urutan itu benar-benar berlaku, bukan sekadar kebetulan.

Tabel di akhir sel adalah temuan penting: pada 10% umur terakhir, FD002 tidak pernah "kritis", sedangkan FD004 sekitar separuhnya. "Kritis" adalah mode kerusakan fan di FD004, bukan tahap yang lebih parah. Ini perlu diputuskan kelompok sebelum dipakai sebagai state RL.

In [ ]:
kol_m = [f'{s}_mean10' for s in SENSOR_AKTIF]
if os.path.exists(f'{MODEL}/health_kmeans.joblib'):
    km_h = joblib.load(f'{MODEL}/health_kmeans.joblib')
else:
    km_h = KMeans(n_clusters=3, n_init=10, random_state=42).fit(fA[kol_m])
    joblib.dump(km_h, f'{MODEL}/health_kmeans.joblib')
F['tahap_idx'] = km_h.predict(F[kol_m])
urut = F[F['kelompok'] == 'A'].groupby('tahap_idx')['cycle'].mean().sort_values().index.tolist()
assert urut == [0, 1, 2], f'cluster belum berurutan sehat/menurun/kritis: {urut}; engine.health akan salah label'
F['tahap'] = F['tahap_idx'].map(dict(enumerate(NAMA_TAHAP)))
akhir = F[F['cycle'] / F.groupby('unit')['cycle'].transform('max') > 0.9]
print('10% umur terakhir, proporsi tahap per subset:')
pd.crosstab(akhir['unit'].str[:5], akhir['tahap'], normalize='index').round(2)

## 9. Artefak `models/v1/`

| File | Isi |
|---|---|
| `rul_model.joblib` | Model terpilih |
| `features.json` | Urutan 43 fitur |
| `rl_config.json` | Batas bin RUL, rumus state (`bin * 3 + tahap`), biaya. **Orang 3 harus memakai rumus yang sama di `rl_env.py`.** |
| `metadata.json` | Versi Python dan library, metrik ringkas. Kunci versi yang sama di `backend/requirements.txt`. |

`kmeans_cond.joblib`, `norm_stats.json`, dan `health_kmeans.joblib` ikut tersimpan bila tadi direproduksi. Folder ini tidak di-commit; salin utuh ke server.

In [ ]:
joblib.dump(m, f'{MODEL}/rul_model.joblib')
json.dump(X, open(f'{MODEL}/features.json', 'w'))
json.dump({'batas_bin': [10, 20, 35, 50, 80], 'label_bin': ['<10', '10-19', '20-34', '35-49', '50-79', '>=80'],
           'n_tahap': 3, 'urutan_tahap': NAMA_TAHAP, 'rumus_state': 'bin * n_tahap + tahap',
           'n_state': 18, 'aksi': ['lanjut', 'rawat'], 'biaya_rawat': 10, 'biaya_gagal': {'50': 50, '100': 100, '200': 200}},
          open(f'{MODEL}/rl_config.json', 'w'), indent=2)
baris = tabel.loc[terpilih]
json.dump({'model_version': 'v1', 'dibuat': datetime.now(timezone(timedelta(hours=7))).isoformat(timespec='seconds'),
           'python': platform.python_version(),
           'library': {p: version(p) for p in ['scikit-learn', 'lightgbm', 'numpy', 'pandas', 'joblib']},
           'model_rul': terpilih,
           'metrik': {'rul_rmse_test': baris['rmse'], 'rul_mae_test': baris['mae'], 'rul_rmse_b': baris['rmse_b'],
                      'silhouette_kondisi': ref.get('silhouette_kondisi')}},
          open(f'{MODEL}/metadata.json', 'w'), indent=2)
print(sorted(os.listdir(MODEL)))

## 10. `predictions.csv` untuk Orang 3

Prediksi setiap siklus untuk mesin kelompok B (training agen) dan C (evaluasi). `rul_aktual` **tidak dipotong**, karena dipakai untuk menghitung siklus terbuang, dan tidak boleh masuk ke state. RMSE di B dan C mirip dengan test, jadi model tidak hanya hafal kelompok A.

In [ ]:
BC = F[F['kelompok'].isin(['B', 'C'])].copy()
BC['rul_pred'] = R.tebak(m, BC[X]).round(2)
BC['rul_aktual'] = BC.groupby('unit')['cycle'].transform('max') - BC['cycle']     # tidak dipotong
BC[['unit', 'cycle', 'kelompok', 'rul_pred', 'rul_aktual', 'tahap']].to_csv(f'{HASIL}/predictions.csv', index=False)
print(BC.groupby('kelompok').apply(lambda d: R.skor(d['rul_aktual'].clip(upper=RUL_CAP), d['rul_pred'])))

## 11. Uji `engine.predict`

Fungsi yang akan dipanggil `/api/prediksi` dijalankan pada 20 mesin test mentah, lalu hasilnya dibandingkan dengan prediksi notebook. Kalau cocok, server dan notebook dijamin memakai pipeline yang sama. Kolom `aksi` dan `q` masih `None` sampai Q-table (`qtable_r*.npy`) tersedia.

In [ ]:
from engine.predict import load, predict
bundle = load(MODEL)
contoh = test[test['unit'].isin(ft_akhir.index[:20])][['unit', 'cycle'] + SETTING + [f's{i}' for i in range(1, 22)]]
keluar = pd.DataFrame(predict(contoh, bundle)).set_index('unit')
banding = R.tebak(m, ft_akhir.loc[keluar.index, X])
assert np.allclose(keluar['rul_pred'], banding, atol=0.06), 'engine.predict berbeda dengan notebook'
print('engine.predict cocok dengan notebook untuk', len(keluar), 'mesin')
keluar.head()